**Data Pre-Processing Steps**

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("/Users/darshantv208gmail.com/Downloads/Mini Proj 2 IIMSTC/Engineering-Novelty-Categorizer-/data_processed/final_dataset_balanced.csv")

In [3]:
data = data.dropna(subset=['clean_text'])
data = data[data['clean_text'].astype(str).str.strip() != ""]
print(f"Rows remaining after removing empty data: {len(data)}")

Rows remaining after removing empty data: 200


In [4]:
def compute_genuine_novelty_score(text):
    if pd.isna(text):
        return 0

    text = str(text).lower()

    breakthrough_keywords = [
        'breakthrough', 'revolutionary', 'unprecedented', 'pioneer',
        'fundamental', 'novelty', 'disruptive', 'paradigm shift'
    ]

    high_keywords = [
        'novel', 'inventive', 'unique', 'innovative', 'original',
        'state-of-the-art', 'sophisticated', 'creative'
    ]

    mod_keywords = [
        'new', 'improved', 'enhanced', 'advanced', 'optimized',
        'efficient', 'modern', 'superior', 'augmented'
    ]

    neg_keywords = [
        'conventional', 'prior art', 'standard', 'known', 'traditional',
        'existing', 'previous', 'typical', 'ordinary', 'common'
    ]

    score = (sum(text.count(kw) for kw in breakthrough_keywords) * 5.0 +
                sum(text.count(kw) for kw in high_keywords) * 3.0 +
                sum(text.count(kw) for kw in mod_keywords) * 1.5 -
                sum(text.count(kw) for kw in neg_keywords) * 1.0)

    words = text.split()
    if len(words) > 0:
        diversity = len(set(words)) / len(words)
        score += (len(words) / 100) * diversity

    return score


In [5]:
def assign_tier(score):
    if score >= 15:
        return 3
    elif score >= 5:
        return 2
    elif score >= 1:
        return 1
    else:
        return 0

In [6]:
labeled_data = data[['clean_text', 'novelty tier']]

In [7]:
labeled_data.to_csv("labeled_patents.csv", index=False)
print(f"File saved successfully as {'labeled_patents.csv'}")
print("Tier Distribution:")
print(labeled_data['novelty tier'].value_counts().sort_index())

File saved successfully as labeled_patents.csv
Tier Distribution:
novelty tier
0    50
1    50
2    50
3    50
Name: count, dtype: int64


In [8]:
unbalenced_data= pd.read_csv("labeled_patents.csv")

In [9]:
balanced_data = unbalenced_data.groupby('novelty tier').apply(
    lambda x: x.sample(n=50, random_state=42)
).reset_index(drop=True)

/var/folders/rz/33t8gsy50jvbn9wj3vmf07b00000gn/T/ipykernel_2871/2926203311.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_data = unbalenced_data.groupby('novelty tier').apply(


In [10]:
balanced_data.to_csv('balanced_data.csv', index=False)

print("\nDistribution after balancing:")
print(balanced_data['novelty tier'].value_counts().sort_index())
print(f"\nBalanced data successfully saved to {'balanced_data.csv'}")


Distribution after balancing:
novelty tier
0    50
1    50
2    50
3    50
Name: count, dtype: int64

Balanced data successfully saved to balanced_data.csv


In [11]:
balanced_dataset = pd.read_csv('balanced_data.csv')
balanced_dataset_shuffled = balanced_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
balanced_dataset_shuffled.to_csv('final_dataset_balanced.csv', index=False)
print(f"Shuffled data saved successfully to final_dataset_balanced.csv")

Shuffled data saved successfully to final_dataset_balanced.csv


In [12]:
X = balanced_dataset_shuffled['clean_text']
Y = balanced_dataset_shuffled['novelty tier']

In [13]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(X, Y,test_size = 0.2, random_state = 42)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=1000) # You can adjust max_features

# Fit and transform the training data
X_train_vec = tfidf_vectorizer.fit_transform(X_train)

# Transform the test data
X_test_vec = tfidf_vectorizer.transform(X_test)

**Machine Learning Model of SVM** 

In [15]:
# Importing Modules and Metrics

from sklearn.svm import SVC
from sklearn.metrics import f1_score, mean_squared_error, accuracy_score, precision_score, recall_score, classification_report

In [16]:
# Training the Model

svm_classifier = SVC(kernel='linear', random_state=42)
svm_classifier.fit(X_train_vec, Y_train)

SVC(kernel='linear', random_state=42)

In [17]:
# Making Predictons

prediction_svm = svm_classifier.predict(X_test_vec)

In [18]:
micro_f1 = f1_score(Y_test, prediction_svm, average='micro')
macro_f1 = f1_score(Y_test, prediction_svm, average='macro')
mse = mean_squared_error(Y_test, prediction_svm)
accuracy = accuracy_score(Y_test, prediction_svm)
precision = precision_score(Y_test, prediction_svm, average='weighted')
recall = recall_score(Y_test, prediction_svm, average='weighted')

print(f"SVM Model Performance:\n")
print(f"Micro F1 Score: {micro_f1:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted): {recall:.4f}\n")

print("Classification Report:\n")
print(classification_report(Y_test, prediction_svm))

print("Overall performance of the SVM model has been evaluated.")

SVM Model Performance:

Micro F1 Score: 0.4000
Macro F1 Score: 0.3943
Mean Squared Error (MSE): 1.5750
Accuracy: 0.4000
Precision (weighted): 0.3944
Recall (weighted): 0.4000

Classification Report:

              precision    recall  f1-score   support

           0       0.44      0.36      0.40        11
           1       0.38      0.27      0.32        11
           2       0.33      0.30      0.32        10
           3       0.43      0.75      0.55         8

    accuracy                           0.40        40
   macro avg       0.40      0.42      0.39        40
weighted avg       0.39      0.40      0.38        40

Overall performance of the SVM model has been evaluated.
